# 📧 Spam Email Detection using Python
### NIELIT Gorakhpur — Data Science & ML using Python
**Session:** 2025-2026 | **Guide:** Mr. Prashant Gupta (Scientist)

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string, pickle, warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

print('All libraries imported successfully!')

## Step 2: Load Dataset

In [ ]:
# Download dataset from Kaggle: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset
# Place spam.csv in the data/ folder
df = pd.read_csv('data/spam.csv', encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
print('Shape:', df.shape)
df.head(10)

## Step 3: Data Cleaning

In [ ]:
print('Null values:\n', df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
df.drop_duplicates(inplace=True)

# Encode labels
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})

# Add length features
df['msg_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(x.split()))

print('\nClass Distribution:')
print(df['label'].value_counts())
df.describe()

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Class Distribution: Spam vs Ham', fontsize=16, fontweight='bold')
colors = ['#2196F3', '#F44336']

axes[0].pie(df['label'].value_counts(), labels=['Ham', 'Spam'],
            autopct='%1.1f%%', colors=colors, startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Proportion')

df['label'].value_counts().plot(kind='bar', color=colors, ax=axes[1], edgecolor='white')
axes[1].set_title('Count')
axes[1].set_xticklabels(['Ham', 'Spam'], rotation=0)

plt.tight_layout()
plt.savefig('outputs/1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Message Length Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Message Length Distribution', fontsize=16, fontweight='bold')

for label, color in zip(['ham', 'spam'], ['#2196F3', '#F44336']):
    axes[0].hist(df[df['label']==label]['msg_length'], bins=50, alpha=0.6, label=label.upper(), color=color)
axes[0].legend()
axes[0].set_xlabel('Message Length')

sns.boxplot(data=df, x='label', y='msg_length', palette={'ham':'#2196F3','spam':'#F44336'}, ax=axes[1])

plt.tight_layout()
plt.savefig('outputs/2_message_length.png', dpi=150, bbox_inches='tight')
plt.show()

print('Spam avg length:', round(df[df['label']=="spam"]['msg_length'].mean(),1))
print('Ham  avg length:', round(df[df['label']=="ham"]['msg_length'].mean(),1))

## Step 5: Text Preprocessing

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = ''.join([c for c in text if c not in string.punctuation and not c.isdigit()])
    tokens = text.split()
    tokens = [stemmer.stem(w) for w in tokens if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

df['cleaned_message'] = df['message'].apply(preprocess_text)

print('Sample cleaned messages:')
df[['message', 'cleaned_message']].head(5)

## Step 6: TF-IDF Feature Extraction & Train-Test Split

In [ ]:
X = df['cleaned_message']
y = df['label_encoded']

tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X_transformed = tfidf.fit_transform(X).toarray()
print('Feature matrix shape:', X_transformed.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X_transformed, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')

## Step 7: Model Training — Naive Bayes

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)
print('Model trained successfully!')

## Step 8: Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

print('Accuracy :', round(accuracy_score(y_test, y_pred)*100, 2), '%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Ham','Spam']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.savefig('outputs/4_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9: Save Model

In [ ]:
with open('models/spam_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print('Model saved!')

## Step 10: Live Prediction Demo

In [ ]:
def predict_message(message):
    cleaned = preprocess_text(message)
    vectorized = tfidf.transform([cleaned]).toarray()
    result = model.predict(vectorized)[0]
    prob = model.predict_proba(vectorized)[0]
    label = '🚨 SPAM' if result == 1 else '✅ HAM (Not Spam)'
    return label, round(max(prob)*100, 2)

test_messages = [
    'Congratulations! You have won a FREE iPhone. Click here to claim now!',
    'Hey, are we still meeting at 5pm today?',
    'URGENT: Your account has been suspended. Call 0800-FREE to restore access.',
    'Can you please send me the notes from class?',
]

for msg in test_messages:
    label, conf = predict_message(msg)
    print(f'Message    : {msg[:60]}...')
    print(f'Prediction : {label}  (Confidence: {conf}%)\n')